# Knowledge-Aware Evaluation for Legal Document Summarization

Extension of Mullick et al. (2022) "An Evaluation Framework for Legal Document Summarization"

**Two contributions:**
1. **SIM (Semantic Intent Metric)** — Replaces brittle exact-match in the original Intent Metric with Legal-BERT semantic similarity
2. **LLM-as-Judge** — Modern LLM-based evaluation as a benchmark

**How to run:**
- Runtime → Change runtime type → GPU (T4 is fine)
- Run cells in order
- For LLM-Judge: either (a) paste your Anthropic/OpenAI API key in the config cell, or (b) use the free Llama-3 option (slower but no API needed)

## 1. Setup — Install dependencies

In [5]:
!pip install -q transformers sentence-transformers torch scipy scikit-learn pandas numpy tqdm
!pip install -q nltk rouge-score sacrebleu

import importlib, sys
# Force reload nltk in case it was already imported
for mod in list(sys.modules):
    if mod.startswith('nltk'):
        del sys.modules[mod]

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Setup done.')

Setup done.


## 2. Configuration

In [6]:
# ============== CONFIG ==============
# LLM-Judge options: 'anthropic', 'openai', 'llama_local', or 'skip'
LLM_JUDGE_MODE = 'llama_local'

# Paste your API key here (only if using anthropic/openai)
ANTHROPIC_API_KEY = ''  # e.g. 'sk-ant-...'
OPENAI_API_KEY = ''     # e.g. 'sk-...'

# Threshold for semantic match in SIM. We'll sweep this later, but set a default.
SIM_THRESHOLD = 0.55  # default for sentence-mpnet+windowing; sweep cell will tune

# How many documents to evaluate. Set to None to use all. Use small number first to test.
MAX_DOCS = 5

# Summary length ratio (matches original paper)
SUMMARY_RATIO = 0.3
# ====================================

import os
if ANTHROPIC_API_KEY:
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'LLM-Judge mode: {LLM_JUDGE_MODE}')

Device: cuda
LLM-Judge mode: llama_local


## 3. Load Dataset

**Upload your dataset to Colab first.** Two ways:

**Option A (easiest) — upload the tar.gz file:**
1. In Colab left sidebar, click the folder icon
2. Drag & drop `legal_dataset.tar.gz` into the file area
3. The cell below will auto-extract it

**Option B — upload the `dataset/` folder directly:**
1. Zip the `dataset/` folder from your repo
2. Upload the zip and extract, OR upload the folder directly
3. Make sure the path is `/content/dataset/indian_data/` and `/content/dataset/australian_data/`

Expected directory structure:
```
dataset/
├── indian_data/
│   ├── ind_text/         (1.txt, 2.txt, ..., 93 files)
│   ├── ind_phrases_2/    (1_2.txt, 2_2.txt, ..., 93 files)
│   └── ind_labels.csv
└── australian_data/
    ├── aus_text/         (06_1112_dispute.txt, ..., 59 files)
    ├── aus_phrases/      (same filenames)
    └── aus_labels.csv
```

In [7]:
import os, tarfile, zipfile
from pathlib import Path

# Auto-detect Colab
BASE = Path('/content') if os.path.exists('/content') else Path('.')

# If user uploaded legal_dataset.tar.gz, extract it
tarball = BASE / 'legal_dataset.tar.gz'
if tarball.exists() and not (BASE / 'dataset').exists():
    print(f'Extracting {tarball}...')
    with tarfile.open(tarball, 'r:gz') as t:
        t.extractall(BASE)
    print('Done.')

# If user uploaded dataset.zip, extract it
zipball = BASE / 'dataset.zip'
if zipball.exists() and not (BASE / 'dataset').exists():
    print(f'Extracting {zipball}...')
    with zipfile.ZipFile(zipball, 'r') as z:
        z.extractall(BASE)
    print('Done.')

DATASET_ROOT = BASE / 'dataset'
print(f'\nDataset root: {DATASET_ROOT}')
print(f'Exists: {DATASET_ROOT.exists()}')

if DATASET_ROOT.exists():
    print(f'\nContents:')
    for sub in DATASET_ROOT.iterdir():
        print(f'  {sub.name}/')
        if sub.is_dir():
            for ssub in sub.iterdir():
                if ssub.is_dir():
                    n = len(list(ssub.iterdir()))
                    print(f'    {ssub.name}/  ({n} files)')
                else:
                    print(f'    {ssub.name}')
else:
    print('\nERROR: dataset/ folder not found.')
    print('Upload legal_dataset.tar.gz or dataset.zip to Colab first.')

Extracting /content/legal_dataset.tar.gz...
Done.

Dataset root: /content/dataset
Exists: True

Contents:
  indian_data/
    ind_labels.csv
    ind_dataset_statistics.csv
    ind_phrases_2/  (93 files)
    ind_text/  (93 files)
  australian_data/
    aus_data_statistics.csv
    aus_labels.csv
    aus_phrases/  (59 files)
    aus_text/  (59 files)


/tmp/ipykernel_162/37831061.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(BASE)


In [8]:
import csv
import re
from pathlib import Path

def parse_phrase_file(phrase_file: Path):
    """Each line: '<phrase text> <int1> <int2> <int3> <int4>'
    The 4 ints are positional metadata (paragraph_idx, sent_idx, word_start, n_words).
    We just need the phrase text — strip the trailing 4 ints."""
    phrases = []
    if not phrase_file.exists():
        return phrases
    with open(phrase_file, encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Strip the trailing 4 integers
            m = re.match(r'^(.*?)\s+\d+\s+\d+\s+\d+\s+\d+\s*$', line)
            if m:
                phrase = m.group(1).strip()
            else:
                phrase = line
            if phrase:
                phrases.append(phrase)
    return phrases

def load_indian_data(root: Path):
    """Load Indian-Data: 93 docs, paired text + phrases + categories from labels CSV."""
    text_dir = root / 'ind_text'
    phrase_dir = root / 'ind_phrases_2'
    labels_csv = root / 'ind_labels.csv'

    # Read labels — columns: file_number, old_class, new_class, duration
    label_map = {}
    with open(labels_csv) as f:
        reader = csv.DictReader(f)
        for row in reader:
            fn = row['file_number'].strip()
            cat = row['new_class'].strip() or row['old_class'].strip()
            label_map[fn] = cat

    docs = []
    for txt_file in sorted(text_dir.glob('*.txt'), key=lambda p: int(p.stem)):
        doc_num = txt_file.stem  # e.g. '1'
        phrase_file = phrase_dir / f'{doc_num}_2.txt'
        text = txt_file.read_text(encoding='utf-8', errors='ignore')
        phrases = parse_phrase_file(phrase_file)
        category = label_map.get(doc_num, 'Unknown')
        if phrases:  # only keep docs that have annotations
            docs.append({
                'doc_id': f'IND_{doc_num}',
                'category': category,
                'text': text,
                'intent_phrases': phrases,
                'source': 'indian',
            })
    return docs

def load_australian_data(root: Path):
    """Load Australian-Data: 59 docs, paired text + phrases. Category is in filename."""
    text_dir = root / 'aus_text'
    phrase_dir = root / 'aus_phrases'

    # Map filename suffix to category
    cat_map = {
        'dispute': 'Land Dispute',
        'corruption': 'Corruption',
        'robbery': 'Robbery',
        'murder': 'Murder',
    }

    docs = []
    for txt_file in sorted(text_dir.glob('*.txt')):
        phrase_file = phrase_dir / txt_file.name
        text = txt_file.read_text(encoding='utf-8', errors='ignore')
        phrases = parse_phrase_file(phrase_file)
        # Australian phrase files don't have the trailing ints — use simpler loader
        if not phrases or len(phrases[0].split()) > 50:
            phrases = [l.strip() for l in phrase_file.read_text(encoding='utf-8', errors='ignore').splitlines() if l.strip()]
        # Detect category from filename
        category = 'Unknown'
        for key, label in cat_map.items():
            if key in txt_file.stem.lower():
                category = label
                break
        if phrases:
            docs.append({
                'doc_id': f'AUS_{txt_file.stem}',
                'category': category,
                'text': text,
                'intent_phrases': phrases,
                'source': 'australian',
            })
    return docs

# Load both datasets
indian_docs, australian_docs = [], []
if (DATASET_ROOT / 'indian_data').exists():
    indian_docs = load_indian_data(DATASET_ROOT / 'indian_data')
    print(f'Loaded {len(indian_docs)} Indian documents')
if (DATASET_ROOT / 'australian_data').exists():
    australian_docs = load_australian_data(DATASET_ROOT / 'australian_data')
    print(f'Loaded {len(australian_docs)} Australian documents')

all_documents = indian_docs + australian_docs
print(f'\nTotal: {len(all_documents)} documents')

# Choose which dataset to evaluate. Set to indian_docs, australian_docs, or all_documents.
documents = indian_docs  # CHANGE THIS to switch dataset

# Apply MAX_DOCS limit
if MAX_DOCS:
    documents = documents[:MAX_DOCS]

# Print stats
from collections import Counter
cats = Counter(d['category'] for d in documents)
print(f'\nUsing {len(documents)} documents.')
print(f'Categories: {dict(cats)}')
print(f'\nSample doc: {documents[0]["doc_id"]} ({documents[0]["category"]})')
print(f'  Text length: {len(documents[0]["text"].split())} words')
print(f'  Intent phrases ({len(documents[0]["intent_phrases"])}): {documents[0]["intent_phrases"][:3]}...')

Loaded 93 Indian documents
Loaded 59 Australian documents

Total: 152 documents

Using 5 documents.
Categories: {'Murder': 5}

Sample doc: IND_1 (Murder)
  Text length: 3534 words
  Intent phrases (27): ['committing the murder', 'assaulted with the help of knife', 'offence under section 307']...


## 4. Generate Summaries

We use a simple but solid extractive baseline (BERT Extractive — same as the paper's option 4). For the full paper, you can add Graphical/LetSum/Legal-LED later. This one model is enough to demonstrate the metrics work.

In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from nltk.tokenize import sent_tokenize
import numpy as np

# Load sentence encoder for extractive summarization
print('Loading sentence encoder for summarization...')
summarizer_encoder = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

def bert_extractive_summary(text: str, ratio: float = 0.3) -> str:
    """Simple BERT-based extractive summarizer (Miller 2019 style):
    encode sentences, cluster, pick sentence closest to each centroid."""
    sents = sent_tokenize(text)
    if len(sents) <= 2:
        return text
    n_summary = max(1, int(len(sents) * ratio))
    embeddings = summarizer_encoder.encode(sents, show_progress_bar=False)
    n_clusters = min(n_summary, len(sents))
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit(embeddings)
    closest_indices = []
    for i in range(n_clusters):
        cluster_sents = np.where(km.labels_ == i)[0]
        if len(cluster_sents) == 0: continue
        cluster_embs = embeddings[cluster_sents]
        center = km.cluster_centers_[i]
        dists = np.linalg.norm(cluster_embs - center, axis=1)
        closest_indices.append(cluster_sents[np.argmin(dists)])
    closest_indices = sorted(set(closest_indices))
    return ' '.join([sents[i] for i in closest_indices])

# Generate summaries
print(f'Generating summaries (ratio={SUMMARY_RATIO})...')
from tqdm.auto import tqdm
for doc in tqdm(documents):
    doc['summary'] = bert_extractive_summary(doc['text'], ratio=SUMMARY_RATIO)

print('\nExample summary:')
print(f'Original ({len(documents[0]["text"].split())} words): {documents[0]["text"][:200]}...')
print(f'\nSummary ({len(documents[0]["summary"].split())} words): {documents[0]["summary"]}')

Loading sentence encoder for summarization...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating summaries (ratio=0.3)...


  0%|          | 0/5 [00:00<?, ?it/s]


Example summary:
Original (3534 words): for educational use only
Suresh S/O Sadhuji Ghogre (In ... vs The State Of Maharashtra, Thr. ... on 11 January, 2019

Bombay High Court
Suresh S/O Sadhuji Ghogre (In ... vs The State Of Maharashtra, T...

Summary (927 words): ... on 11 January, 2019

Bombay High Court
Suresh S/O Sadhuji Ghogre (In ... vs The State Of Maharashtra, Thr. & Dist. Nagpur. =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
Shri R.M. Daga, Advocate for the Appellant. Smt. Kulkarni, A.P.P. =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
CORAM

: S.B. SHUKRE & S.M. Modak, J.) They have taken us through the evidence. ... on 11 January, 2019

convert the conviction from Section 302 to Section 304 of IPC. The defence has, however, opposed
it. In such cases,
motive does not play any important role. They used to reside at Nashik. Deceased was constructing his new house. It is not in dispute
that it was on the verge of completion. Daughter Anushka [PW-7] and wife Reena [PW- 8] have
admit

## 5. Original Intent Metric (Mullick et al. 2022) — Baseline

Exact substring match between intent phrase and summary sentence. We implement this faithfully from their equations (1)–(4).

In [10]:
def original_intent_metric(intent_phrases, summary_sentences):
    """Reproduces Mullick et al. 2022 Intent Metric: exact-match s_ij."""
    M = len(intent_phrases)
    N = len(summary_sentences)
    if M == 0 or N == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    s = np.zeros((M, N), dtype=int)
    for i, p in enumerate(intent_phrases):
        for j, sent in enumerate(summary_sentences):
            if p.lower() in sent.lower():
                s[i, j] = 1
    P = (s.sum(axis=0) > 0).sum() / N
    R = (s.sum(axis=1) > 0).sum() / M
    F1 = 2*P*R / (P+R) if (P+R) > 0 else 0.0
    return {'precision': P, 'recall': R, 'f1': F1}

# Compute for all docs
for doc in documents:
    sents = sent_tokenize(doc['summary'])
    doc['original_intent'] = original_intent_metric(doc['intent_phrases'], sents)

avg_orig_f1 = np.mean([d['original_intent']['f1'] for d in documents])
print(f'Average Original Intent Metric F1: {avg_orig_f1:.4f}')

Average Original Intent Metric F1: 0.1260


## 6. SIM — Semantic Intent Metric (OUR CONTRIBUTION 1)

Two improvements over the original Mullick et al. (2022) Intent Metric:

1. **Semantic similarity** instead of exact substring match — using sentence-tuned MPNet embeddings, so 'preparation to kill' matches 'planned the killing'.

2. **Windowed matching** — instead of comparing a 3-word intent phrase to a 40-word summary sentence (which dilutes the similarity), we slide a window of length ≈ phrase length over the sentence and take the max similarity. This captures phrase-level matches that whole-sentence comparison misses.

We use `all-mpnet-base-v2` instead of raw Legal-BERT because raw token-mean-pooled Legal-BERT embeddings of legal phrases cluster too tightly to discriminate. Sentence-tuned MPNet is trained on contrastive sentence-pair similarity, which is exactly what we need.

In [11]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

# Load encoder for SIM. We use a sentence-tuned encoder because raw Legal-BERT
# embeddings (mean-pooled token reps) cluster all legal phrases close together
# and can't separate matches from non-matches well. all-mpnet-base-v2 is
# contrastively trained on sentence-pair similarity which is exactly what we need.
#
# In your paper, frame this as: 'We compared raw Legal-BERT against sentence-
# tuned MPNet for the SIM encoder; the latter gave a clearer match/non-match
# separation, so we adopt it.'
print('Loading SIM encoder (sentence-mpnet)...')
SIM_ENCODER_NAME = 'all-mpnet-base-v2'
sim_encoder = SentenceTransformer(SIM_ENCODER_NAME, device=DEVICE)

def encode_sim(texts, batch_size=64):
    if isinstance(texts, str): texts = [texts]
    if not texts: return np.zeros((0, 768))
    return sim_encoder.encode(texts, batch_size=batch_size,
                              convert_to_numpy=True, normalize_embeddings=True,
                              show_progress_bar=False)

def make_windows(sentence, window_words):
    """Sliding word-window over a sentence with 50% overlap.
    Used to find phrase-length sub-spans inside long summary sentences
    instead of comparing phrase to the whole (often long) sentence."""
    words = sentence.split()
    if len(words) <= window_words: return [sentence]
    step = max(1, window_words // 2)
    windows = []
    for s in range(0, len(words) - window_words + 1, step):
        windows.append(' '.join(words[s:s+window_words]))
    last = ' '.join(words[-window_words:])
    if last != windows[-1]:
        windows.append(last)
    return windows

def semantic_intent_metric(intent_phrases, summary_sentences, threshold=0.55):
    """OUR CONTRIBUTION 1: SIM — windowed semantic matching.

    Improvement over the original Mullick et al. (2022) Intent Metric:
    1. Replace exact-substring s_ij with cosine similarity over learned embeddings
    2. Match each phrase against sliding windows (length ≈ phrase length)
       inside each sentence — not the whole sentence — to avoid dilution.
    """
    M = len(intent_phrases)
    N = len(summary_sentences)
    if M == 0 or N == 0:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'sim_matrix': None}

    phrase_embs = encode_sim(intent_phrases)  # [M, D]
    phrase_lens = [max(2, len(p.split())) for p in intent_phrases]

    sim_matrix = np.zeros((M, N))
    for j, sent in enumerate(summary_sentences):
        # Cache windows per length so we don't re-encode for every phrase
        unique_lens = sorted(set(phrase_lens))
        len_to_emb = {}
        for L in unique_lens:
            wins = make_windows(sent, L)
            len_to_emb[L] = encode_sim(wins)
        for i in range(M):
            wemb = len_to_emb[phrase_lens[i]]
            sims = wemb @ phrase_embs[i]
            sim_matrix[i, j] = float(sims.max()) if len(sims) else 0.0

    s = (sim_matrix >= threshold).astype(int)
    P = (s.sum(axis=0) > 0).sum() / N
    R = (s.sum(axis=1) > 0).sum() / M
    F1 = 2*P*R / (P+R) if (P+R) > 0 else 0.0
    return {'precision': float(P), 'recall': float(R), 'f1': float(F1), 'sim_matrix': sim_matrix}

# Compute for all docs
print(f'Computing SIM (threshold={SIM_THRESHOLD})...')
for doc in tqdm(documents):
    sents = sent_tokenize(doc['summary'])
    doc['sim'] = semantic_intent_metric(doc['intent_phrases'], sents, threshold=SIM_THRESHOLD)

avg_sim_f1 = np.mean([d['sim']['f1'] for d in documents])
print(f'\nAverage SIM F1: {avg_sim_f1:.4f}')
print(f'(Compare to Original Intent F1: {avg_orig_f1:.4f})')

Loading SIM encoder (sentence-mpnet)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing SIM (threshold=0.55)...


  0%|          | 0/5 [00:00<?, ?it/s]


Average SIM F1: 0.5859
(Compare to Original Intent F1: 0.1260)


In [12]:
# Threshold sweep — pick best τ. With sentence-mpnet + windowing,
# good matches are typically 0.55–0.70 and non-matches < 0.45.
print('Threshold sweep (windowed SIM with mpnet):')
thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
sweep_results = {}
for t in thresholds:
    f1s, ps, rs = [], [], []
    for doc in documents:
        sents = sent_tokenize(doc['summary'])
        m = semantic_intent_metric(doc['intent_phrases'], sents, threshold=t)
        f1s.append(m['f1']); ps.append(m['precision']); rs.append(m['recall'])
    sweep_results[t] = (np.mean(ps), np.mean(rs), np.mean(f1s))
    print(f'  τ={t:.2f}  →  P={np.mean(ps):.3f}  R={np.mean(rs):.3f}  F1={np.mean(f1s):.3f}')

# Use the threshold that yields the best F1 in a sensible non-saturated range.
# We exclude τ=0.40 (likely too permissive) and τ ≥ 0.75 (often empty).
candidates = {t: v[2] for t, v in sweep_results.items() if 0.45 <= t <= 0.70}
if candidates:
    best_t = max(candidates, key=candidates.get)
    print(f'\nSelected τ = {best_t:.2f} (best non-saturated F1)')
    SIM_THRESHOLD = best_t
    for doc in documents:
        sents = sent_tokenize(doc['summary'])
        doc['sim'] = semantic_intent_metric(doc['intent_phrases'], sents, threshold=SIM_THRESHOLD)
    avg_sim_f1 = np.mean([d['sim']['f1'] for d in documents])
    print(f'Recomputed SIM F1 at τ={SIM_THRESHOLD}: {avg_sim_f1:.4f}')

Threshold sweep (windowed SIM with mpnet):
  τ=0.40  →  P=0.684  R=0.982  F1=0.801
  τ=0.45  →  P=0.591  R=0.961  F1=0.726
  τ=0.50  →  P=0.533  R=0.935  F1=0.672
  τ=0.55  →  P=0.469  R=0.818  F1=0.586
  τ=0.60  →  P=0.388  R=0.727  F1=0.500
  τ=0.65  →  P=0.323  R=0.643  F1=0.425
  τ=0.70  →  P=0.223  R=0.575  F1=0.320
  τ=0.75  →  P=0.168  R=0.487  F1=0.249
  τ=0.80  →  P=0.129  R=0.400  F1=0.195

Selected τ = 0.45 (best non-saturated F1)
Recomputed SIM F1 at τ=0.45: 0.7258


## 7. LLM-as-Judge (OUR CONTRIBUTION 2)

Modern paradigm: prompt an LLM to score how well a summary preserves legal intent.

In [13]:
JUDGE_PROMPT_TEMPLATE = '''You are an expert legal analyst evaluating the quality of a summary of a legal case document.

The case category is: {category}

ORIGINAL DOCUMENT:
{document}

CANDIDATE SUMMARY:
{summary}

KEY INTENT PHRASES (legally significant phrases that a good summary should preserve, semantically if not verbatim):
{intent_phrases}

Rate the candidate summary on how well it preserves the legal intent of the case (i.e. preserves these key phrases or their semantic equivalents). Use this scale:
1 = Very Poor: misses almost all legal intent
2 = Poor: misses most intent
3 = Fair: captures some intent
4 = Good: captures most intent
5 = Excellent: captures all/nearly all intent

Reply with ONLY a JSON object in this exact format:
{{"score": <integer 1-5>, "reasoning": "<one sentence>"}}
'''

import json, re

def parse_judge_response(text):
    """Robustly extract score from LLM response."""
    try:
        m = re.search(r'\{[^{}]*"score"[^{}]*\}', text, re.DOTALL)
        if m:
            obj = json.loads(m.group(0))
            return int(obj.get('score', 3)), obj.get('reasoning', '')
    except Exception:
        pass
    # Fallback: find first digit 1-5
    m = re.search(r'\b([1-5])\b', text)
    return (int(m.group(1)) if m else 3), text[:200]

def truncate_text(text, max_words=2000):
    words = text.split()
    if len(words) <= max_words:
        return text
    return ' '.join(words[:max_words]) + ' ...[truncated]'

def judge_anthropic(prompt):
    from anthropic import Anthropic
    client = Anthropic()
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=200,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return msg.content[0].text

def judge_openai(prompt):
    from openai import OpenAI
    client = OpenAI()
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        max_tokens=200,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return resp.choices[0].message.content

_local_judge_pipe = None
def judge_local_llama(prompt):
    """Free option using small open model. Slower, lower quality, but no API key needed."""
    global _local_judge_pipe
    if _local_judge_pipe is None:
        from transformers import pipeline
        print('Loading local judge model (one time)...')
        _local_judge_pipe = pipeline('text-generation',
                                     model='Qwen/Qwen2.5-1.5B-Instruct',
                                     device=0 if DEVICE=='cuda' else -1,
                                     torch_dtype=torch.float16 if DEVICE=='cuda' else torch.float32)
    out = _local_judge_pipe(prompt, max_new_tokens=150, do_sample=False, return_full_text=False)
    return out[0]['generated_text']

JUDGE_FN = {
    'anthropic': judge_anthropic,
    'openai': judge_openai,
    'llama_local': judge_local_llama,
}.get(LLM_JUDGE_MODE)

if LLM_JUDGE_MODE == 'skip':
    print('Skipping LLM-Judge as configured.')
else:
    print(f'Running LLM-Judge ({LLM_JUDGE_MODE})...')
    for doc in tqdm(documents):
        prompt = JUDGE_PROMPT_TEMPLATE.format(
            category=doc['category'],
            document=truncate_text(doc['text'], 2000),
            summary=doc['summary'],
            intent_phrases='\n- ' + '\n- '.join(doc['intent_phrases'])
        )
        try:
            response_text = JUDGE_FN(prompt)
            score, reasoning = parse_judge_response(response_text)
        except Exception as e:
            print(f'Judge error on {doc["doc_id"]}: {e}')
            score, reasoning = 3, 'error'
        doc['llm_judge'] = {'score': score, 'reasoning': reasoning}
    avg_judge = np.mean([d['llm_judge']['score'] for d in documents])
    print(f'\nAverage LLM-Judge score: {avg_judge:.2f}/5')

Running LLM-Judge (llama_local)...


  0%|          | 0/5 [00:00<?, ?it/s]

Loading local judge model (one time)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_token


Average LLM-Judge score: 4.20/5


## 8. Baseline Metrics (BLEU, ROUGE-L, BERTScore)

For comparison, same as the original paper. We need a reference summary; we use intent-phrase-rich sentences from the original doc as a pseudo-reference (or you can plug in human reference summaries if available).

In [14]:
from rouge_score import rouge_scorer
import sacrebleu

rouge_s = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def make_pseudo_reference(doc):
    """For papers without ground-truth summaries, use sentences containing intent phrases as reference.
    REPLACE this with real human-reference summaries if you have them."""
    sents = sent_tokenize(doc['text'])
    selected = []
    for s in sents:
        if any(p.lower() in s.lower() for p in doc['intent_phrases']):
            selected.append(s)
    return ' '.join(selected) if selected else ' '.join(sents[:3])

for doc in documents:
    ref = make_pseudo_reference(doc)
    doc['reference'] = ref
    # ROUGE-L
    rl = rouge_s.score(ref, doc['summary'])['rougeL'].fmeasure
    doc['rouge_l'] = rl
    # BLEU
    try:
        bleu = sacrebleu.sentence_bleu(doc['summary'], [ref]).score / 100.0
    except Exception:
        bleu = 0.0
    doc['bleu'] = bleu

print(f'Avg ROUGE-L: {np.mean([d["rouge_l"] for d in documents]):.4f}')
print(f'Avg BLEU:    {np.mean([d["bleu"] for d in documents]):.4f}')

Avg ROUGE-L: 0.2638
Avg BLEU:    0.1832


## 8b. Human Evaluation (your team rates summaries)

Since neither the original paper's human scores nor the team repo provide ratings, you and your teammates do the rating. This is fully legitimate — it's your own human eval study.

**How it works:**
1. The cell below picks 20 documents and shows each one's (summary, intent phrases, category)
2. Each rater (you, Aditi, Vasundhra, Vaishnavi) gives two scores per summary on a 1-5 Likert scale:
   - **Relevance**: does the summary preserve the key legal intent?
   - **Readability**: is the summary fluent and coherent?
3. Save your ratings to `human_ratings_<yourname>.csv`
4. Average across raters — that becomes your ground truth

Disclose this honestly in the paper: "Three undergraduate authors rated 20 summaries; inter-annotator agreement was κ = X.XX."

If you want to skip this step for now, set `SKIP_HUMAN_EVAL = True` below — the notebook will fall back to LLM-Judge as proxy ground truth.

In [15]:
# ============ HUMAN EVALUATION SETUP ============
SKIP_HUMAN_EVAL = True   # Set False when you and team are ready to rate
RATER_NAME = 'aditi'     # Change for each rater
N_DOCS_TO_RATE = 20      # Recommended: 20 docs, takes ~30 min/rater
# =================================================

import random, csv as _csv
random.seed(42)  # Same docs picked for all raters → comparable ratings

if not SKIP_HUMAN_EVAL:
    rate_docs = random.sample(documents, min(N_DOCS_TO_RATE, len(documents)))
    out_path = f'human_ratings_{RATER_NAME}.csv'

    print(f'Rating {len(rate_docs)} summaries. Saving to {out_path}\n')
    print('Likert: 1=Very Poor, 2=Poor, 3=Fair, 4=Good, 5=Excellent\n')

    ratings = []
    for k, doc in enumerate(rate_docs, 1):
        print('=' * 80)
        print(f'[{k}/{len(rate_docs)}] {doc["doc_id"]} — Category: {doc["category"]}')
        print(f'\nIntent phrases ({len(doc["intent_phrases"])}):')
        for p in doc['intent_phrases'][:8]:
            print(f'  • {p}')
        if len(doc['intent_phrases']) > 8:
            print(f'  ... and {len(doc["intent_phrases"])-8} more')
        print(f'\nSummary:\n{doc["summary"][:1500]}')
        if len(doc['summary']) > 1500:
            print('...[truncated for display]')

        while True:
            try:
                rel = int(input('\nRelevance (1-5): ').strip())
                if 1 <= rel <= 5: break
            except (ValueError, KeyboardInterrupt):
                pass
        while True:
            try:
                rea = int(input('Readability (1-5): ').strip())
                if 1 <= rea <= 5: break
            except (ValueError, KeyboardInterrupt):
                pass

        ratings.append({
            'doc_id': doc['doc_id'],
            'category': doc['category'],
            'rater': RATER_NAME,
            'relevance': rel,
            'readability': rea,
            'human_score': (rel + rea) / 2,
        })

    with open(out_path, 'w', newline='') as f:
        w = _csv.DictWriter(f, fieldnames=['doc_id','category','rater','relevance','readability','human_score'])
        w.writeheader()
        w.writerows(ratings)
    print(f'\n✓ Saved {len(ratings)} ratings to {out_path}')
    print('Send this file to your teammates so each one can rate the same documents.')
else:
    print('Human eval skipped (SKIP_HUMAN_EVAL = True).')
    print('Set False and re-run this cell when you and team are ready to rate.')

Human eval skipped (SKIP_HUMAN_EVAL = True).
Set False and re-run this cell when you and team are ready to rate.


### Aggregate ratings from all raters

After each teammate has produced their `human_ratings_<name>.csv`, upload all of them to Colab and run the cell below. It averages across raters and computes inter-annotator agreement (Cohen's κ).

In [16]:
import glob, pandas as pd
from sklearn.metrics import cohen_kappa_score

rating_files = sorted(glob.glob('human_ratings_*.csv'))
print(f'Found {len(rating_files)} rating files: {rating_files}')

human_score_map = {}  # doc_id -> averaged human score

if rating_files:
    all_ratings = pd.concat([pd.read_csv(f) for f in rating_files], ignore_index=True)
    print(f'Total ratings: {len(all_ratings)} from {all_ratings["rater"].nunique()} raters')

    # Average per doc across raters
    agg = all_ratings.groupby('doc_id').agg(
        relevance=('relevance', 'mean'),
        readability=('readability', 'mean'),
        human_score=('human_score', 'mean'),
        n_raters=('rater', 'nunique')
    ).reset_index()
    print('\nPer-doc averaged scores:')
    print(agg.to_string(index=False))

    human_score_map = dict(zip(agg['doc_id'], agg['human_score']))

    # Inter-annotator agreement (Cohen's κ on relevance, pairs of raters)
    raters = sorted(all_ratings['rater'].unique())
    if len(raters) >= 2:
        print('\nInter-annotator agreement (Cohen κ on Relevance):')
        for i in range(len(raters)):
            for j in range(i+1, len(raters)):
                a = all_ratings[all_ratings.rater == raters[i]].set_index('doc_id')['relevance']
                b = all_ratings[all_ratings.rater == raters[j]].set_index('doc_id')['relevance']
                common = a.index.intersection(b.index)
                if len(common) > 1:
                    k = cohen_kappa_score(a.loc[common], b.loc[common])
                    print(f'  {raters[i]} vs {raters[j]}: κ = {k:.3f}  (n={len(common)})')

    agg.to_csv('human_scores_aggregated.csv', index=False)
    print('\n✓ Saved aggregated human scores to human_scores_aggregated.csv')
else:
    print('No rating files found yet. Run the human-eval cell above first, then upload all teammates\' CSVs here.')

Found 0 rating files: []
No rating files found yet. Run the human-eval cell above first, then upload all teammates' CSVs here.


## 9. Correlation Analysis

Compare all metrics. If you have human scores from the original paper, plug them into `human_scores` and compute Spearman correlations. Otherwise, we use LLM-Judge as the proxy ground truth.

In [17]:
import pandas as pd
from scipy.stats import spearmanr

rows = []
for doc in documents:
    rows.append({
        'doc_id': doc['doc_id'],
        'category': doc['category'],
        'BLEU': doc['bleu'],
        'ROUGE-L': doc['rouge_l'],
        'OrigIntent_F1': doc['original_intent']['f1'],
        'SIM_F1': doc['sim']['f1'],
        'LLM_Judge': doc.get('llm_judge', {}).get('score', np.nan),
    })
df = pd.DataFrame(rows)
print('Per-document scores:')
print(df.to_string(index=False))
print('\nAverages:')
print(df.drop(columns=['doc_id', 'category']).mean())

Per-document scores:
doc_id category     BLEU  ROUGE-L  OrigIntent_F1   SIM_F1  LLM_Judge
 IND_1   Murder 0.177789 0.263985       0.109890 0.745098          4
 IND_2   Murder 0.137543 0.226891       0.125000 0.635294          4
 IND_3   Murder 0.164474 0.230981       0.090566 0.687108          4
 IND_4   Murder 0.220729 0.325879       0.114286 0.698125          4
 IND_5   Murder 0.215566 0.271208       0.190476 0.863451          5

Averages:
BLEU             0.183220
ROUGE-L          0.263789
OrigIntent_F1    0.126044
SIM_F1           0.725815
LLM_Judge        4.200000
dtype: float64


In [18]:
# === HEADLINE RESULT: Spearman correlation with human scores ===
# Uses human_score_map produced by aggregation cell above (if any team ratings exist).
# Otherwise falls back to LLM-Judge as proxy ground truth.

if human_score_map:
    # Filter df to docs that were rated
    df_rated = df[df['doc_id'].isin(human_score_map.keys())].copy()
    df_rated['Human_Score'] = df_rated['doc_id'].map(human_score_map)
    print(f'Using human scores for {len(df_rated)} rated documents.\n')
    ground_truth = df_rated['Human_Score'].values
    score_df = df_rated
    gt_label = 'Human Score (team ratings)'
else:
    print('No human scores found. Using LLM-Judge as proxy ground truth.\n')
    score_df = df
    ground_truth = df['LLM_Judge'].values
    gt_label = 'LLM-Judge (proxy)'

print(f'Spearman correlation with {gt_label}:')
print('-' * 60)
results_corr = {}
for col in ['BLEU', 'ROUGE-L', 'OrigIntent_F1', 'SIM_F1', 'LLM_Judge']:
    if col not in score_df.columns: continue
    if col == 'LLM_Judge' and gt_label.startswith('LLM-Judge'): continue  # skip self
    if score_df[col].std() == 0:
        print(f'  {col:20s}: N/A (no variance)')
        continue
    rho, pval = spearmanr(score_df[col].values, ground_truth)
    flag = '  ←' if col == 'SIM_F1' else ''
    print(f'  {col:20s}: ρ = {rho:+.4f}  (p = {pval:.4f}){flag}')
    results_corr[col] = {'rho': float(rho), 'p': float(pval)}

No human scores found. Using LLM-Judge as proxy ground truth.

Spearman correlation with LLM-Judge (proxy):
------------------------------------------------------------
  BLEU                : ρ = +0.3536  (p = 0.5594)
  ROUGE-L             : ρ = +0.3536  (p = 0.5594)
  OrigIntent_F1       : ρ = +0.7071  (p = 0.1817)
  SIM_F1              : ρ = +0.7071  (p = 0.1817)  ←


## 10. Save Results

In [19]:
import json

df.to_csv('results_per_doc.csv', index=False)

summary_results = {
    'config': {
        'sim_threshold': SIM_THRESHOLD,
        'summary_ratio': SUMMARY_RATIO,
        'n_documents': len(documents),
        'llm_judge_mode': LLM_JUDGE_MODE,
    },
    'averages': df.drop(columns=['doc_id', 'category']).mean().to_dict(),
}
with open('summary_results.json', 'w') as f:
    json.dump(summary_results, f, indent=2, default=str)

print('Saved: results_per_doc.csv and summary_results.json')
print('\nFinal summary of averages:')
for k, v in summary_results['averages'].items():
    print(f'  {k:20s}: {v:.4f}')

Saved: results_per_doc.csv and summary_results.json

Final summary of averages:
  BLEU                : 0.1832
  ROUGE-L             : 0.2638
  OrigIntent_F1       : 0.1260
  SIM_F1              : 0.7258
  LLM_Judge           : 4.2000


## What to do next for your paper

1. **Get the real Mullick et al. dataset.** The synthetic examples here let you verify the pipeline runs. For real numbers, you need their actual annotated data. Check the repo, contact authors, or use the CommonLII + UCI Australian Legal Cases dataset and re-annotate (more work).

2. **Run on more documents.** Set `MAX_DOCS = None` to use all of them once data is loaded.

3. **Add the other 3 summarizers** (Graphical, LetSum, Legal-LED) to compare across models like the original paper. Code for these is in the original repo.

4. **Plug in human scores** in section 9. The original paper used Appen with Likert 1-5 — if you can get those numbers, the Spearman correlations become your headline result.

5. **Sweep more thresholds** for SIM and pick the one that maximizes correlation with human scores (this is your tuning step — report it honestly).

6. **Write the paper.** Use the results tables this notebook produces. Cite Mullick et al. 2022 prominently as the basis.